In [16]:
from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")

db = client["kopi_nusantara"]

koleksi_menu = db["menu"]

print("Koneksi ke MongoDB Berhasil")
print("Daftar database yang ada:", client.list_database_names())

spark = SparkSession.builder.appName("Pertemuan9-MongoDBSpark").master("local[*]") .getOrCreate()

Koneksi ke MongoDB Berhasil
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


In [19]:
koleksi_menu.insert_many([
    {"nama_menu": "Espresso Single", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 20},
    {"nama_menu": "Americano Hot", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 4, "stok": 25},
    {"nama_menu": "Caffe Latte", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 2, "stok": 30},
    {"nama_menu": "Cappuccino", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 3, "stok": 18},
    {"nama_menu": "Kopi Susu Gula Aren", "kategori": "Kopi", "harga": 25000, "tingkat_kepahitan": 2, "stok": 40},
    {"nama_menu": "Cold Brew", "kategori": "Kopi", "harga": 30000, "tingkat_kepahitan": 4, "stok": 15},
    {"nama_menu": "Caramel Macchiato", "kategori": "Kopi", "harga": 32000, "tingkat_kepahitan": 1, "stok": 22},
    {"nama_menu": "Mocha Latte", "kategori": "Kopi", "harga": 30000, "tingkat_kepahitan": 2, "stok": 12},
    {"nama_menu": "Nasi Goreng Espresso", "kategori": "Makanan", "harga": 35000, "pedas_level": 2, "stok": 15},
    {"nama_menu": "Mie Goreng Pedas Manis", "kategori": "Makanan", "harga": 25000, "pedas_level": 3, "stok": 25},
    {"nama_menu": "Ayam Geprek Sambal Korek", "kategori": "Makanan", "harga": 28000, "pedas_level": 3, "stok": 20},
    {"nama_menu": "French Fries Original", "kategori": "Makanan", "harga": 20000, "pedas_level": 0, "stok": 35},
    {"nama_menu": "Toast Kaya Butter", "kategori": "Makanan", "harga": 22000, "pedas_level": 0, "stok": 18},
    {"nama_menu": "Spaghetti Bolognese", "kategori": "Makanan", "harga": 38000, "pedas_level": 1, "stok": 10},
    {"nama_menu": "Cireng Bumbu Rujak", "kategori": "Makanan", "harga": 18000, "pedas_level": 2, "stok": 30}
])

print("data berhasil ditambahkan")
print("Jumlah dokumen di collection 'produk' : ", koleksi_produk.count_documents({}))

data berhasil ditambahkan
Jumlah dokumen di collection 'produk' :  15


In [44]:
print("Kopi dengan tingkat kepahitan diatas 3: ")
for k in koleksi_menu.find({"tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {k['nama_menu']} | Tingkat Kepahitan {k['tingkat_kepahitan']}")

queryharga = {
    "harga": {"$gte": 15000, "$lte": 30000}
}

hargaantara = list(koleksi_menu.find(queryharga))

print("\nSeluruh menu dengan harga diantara 15k-30k:")
for m in koleksi_menu.find(queryharga):
    print(f"  - {m['nama_menu']} | dengan harga Rp.{m['harga']}")


querypedas = {
    "pedas_level": {"$exists": False}
}

print("\nmenu yang tidak memiliki field pedas")
for p in koleksi_menu.find(querypedas):
    print(f"  - {p['nama_menu']} | Kategori {p['kategori']}")

Kopi dengan tingkat kepahitan diatas 3: 
  - Espresso Single | Tingkat Kepahitan 5
  - Americano Hot | Tingkat Kepahitan 4
  - Cold Brew | Tingkat Kepahitan 4

Seluruh menu dengan harga diantara 15k-30k:
  - Espresso Single | dengan harga Rp.18000
  - Americano Hot | dengan harga Rp.22000
  - Caffe Latte | dengan harga Rp.28000
  - Cappuccino | dengan harga Rp.28000
  - Kopi Susu Gula Aren | dengan harga Rp.25000
  - Cold Brew | dengan harga Rp.30000
  - Mocha Latte | dengan harga Rp.30000
  - Mie Goreng Pedas Manis | dengan harga Rp.25000
  - Ayam Geprek Sambal Korek | dengan harga Rp.28000
  - French Fries Original | dengan harga Rp.20000
  - Toast Kaya Butter | dengan harga Rp.22000
  - Cireng Bumbu Rujak | dengan harga Rp.18000

menu yang tidak memiliki field pedas
  - Espresso Single | Kategori Kopi
  - Americano Hot | Kategori Kopi
  - Caffe Latte | Kategori Kopi
  - Cappuccino | Kategori Kopi
  - Kopi Susu Gula Aren | Kategori Kopi
  - Cold Brew | Kategori Kopi
  - Caramel Macch

In [52]:
koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

print("Data Berhasil DiUpdate")
for p in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status" : 1}):
    print(f"  - {p['nama_menu']}: {p['status']}")

koleksi_menu.update_many(
    {"nama_menu": {"$in": ["Mocha Latte", "Cappuccino"]}},
    {"$set": {"status": "Habis"}}
)

print("\nData Berhasil DiUpdate")
for p in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status" : 1}):
    print(f"  - {p['nama_menu']}: {p['status']}")

Data Berhasil DiUpdate
  - Espresso Single: Tersedia
  - Americano Hot: Tersedia
  - Caffe Latte: Tersedia
  - Cappuccino: Tersedia
  - Kopi Susu Gula Aren: Tersedia
  - Cold Brew: Tersedia
  - Caramel Macchiato: Tersedia
  - Mocha Latte: Tersedia
  - Nasi Goreng Espresso: Tersedia
  - Mie Goreng Pedas Manis: Tersedia
  - Ayam Geprek Sambal Korek: Tersedia
  - French Fries Original: Tersedia
  - Toast Kaya Butter: Tersedia
  - Spaghetti Bolognese: Tersedia
  - Cireng Bumbu Rujak: Tersedia

Data Berhasil DiUpdate
  - Espresso Single: Tersedia
  - Americano Hot: Tersedia
  - Caffe Latte: Tersedia
  - Cappuccino: Habis
  - Kopi Susu Gula Aren: Tersedia
  - Cold Brew: Tersedia
  - Caramel Macchiato: Tersedia
  - Mocha Latte: Habis
  - Nasi Goreng Espresso: Tersedia
  - Mie Goreng Pedas Manis: Tersedia
  - Ayam Geprek Sambal Korek: Tersedia
  - French Fries Original: Tersedia
  - Toast Kaya Butter: Tersedia
  - Spaghetti Bolognese: Tersedia
  - Cireng Bumbu Rujak: Tersedia


In [58]:
dokumen_mongo = list(koleksi_menu.find({}))

df_pandas = pd.DataFrame(dokumen_mongo)

df_pandas["_id"] = df_pandas["_id"].astype(str)

df_spark_menu = spark.createDataFrame(df_pandas)
df_spark_menu.printSchema()
df_spark_menu.show()

df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_produk")
).show()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- Status: string (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

+--------------------+--------------------+--------+-----+-----------------+----+--------+--------+-----------+
|                 _id|           nama_menu|kategori|harga|tingkat_kepahitan|stok|  Status|  status|pedas_level|
+--------------------+--------------------+--------+-----+-----------------+----+--------+--------+-----------+
|6ae36d4a4b5d734bd...|     Espresso Single|    Kopi|18000|              5.0|  20|Tersedia|Tersedia|        NaN|
|6ae36d4a4b5d734bd...|       Americano Hot|    Kopi|22000|              4.0|  25|Tersedia|Tersedia|        NaN|
|6ae36d4a4b5d734bd...|         Caffe Latte|    Kopi|28000|              2.0|  30|

[Stage 12:=======================>                                 (5 + 7) / 12]

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
|    Kopi|           26625.0|            8|
| Makanan|26571.428571428572|            7|
+--------+------------------+-------------+

